In [ ]:
# Projeto: Análise de Qualidade do Ar - Brasil
# Fonte: OpenAQ API v3
# Autor: Beatriz Dias
# Status: Em desenvolvimento
#
# O que esse notebook faz até aqui:
# - Conecta na API da OpenAQ
# - Busca estações de monitoramento no Brasil
# - Retorna sensores e poluentes da estação de Osasco/SP

In [6]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import os

# Carrega as variáveis do arquivo .env  
load_dotenv("/Users/beatrizdias/Desktop/trabalhos/qualidade_ar_brasil/.env")

API_KEY = os.getenv("OPENAQ_API_KEY")
BASE_URL = "https://api.openaq.org/v3"

HEADERS = {"X-API-Key": API_KEY} 

print("Ambiente configurado com sucesso!") 
print(f"Chave carregada: {API_KEY[:8]}...")  # mostra só os primeiros caracteres por segurança  

Ambiente configurado com sucesso!
Chave carregada: 16a17a1f...


In [9]:
import os
print("Caminho atual:", os.getcwd())

Caminho atual: /Users/beatrizdias/Desktop/trabalhos


In [10]:
def buscar_estacoes(pais="BR", limite=10):
    url = f"{BASE_URL}/locations"
    
    parametros = {
        "country": pais,
        "limit": limite
    }
    
    resposta = requests.get(url, headers=HEADERS, params=parametros)
    
    print(f"Status da requisição: {resposta.status_code}")
    
    return resposta.json()

# Busca as estações e mostra o resultado bruto
dados_brutos = buscar_estacoes()
print(dados_brutos)

Status da requisição: 200
{'meta': {'name': 'openaq-api', 'website': '/', 'page': 1, 'limit': 10, 'found': '>10'}, 'results': [{'id': 3, 'name': 'NMA - Nima', 'locality': None, 'timezone': 'Africa/Accra', 'country': {'id': 152, 'code': 'GH', 'name': 'Ghana'}, 'owner': {'id': 4, 'name': 'Unknown Governmental Organization'}, 'provider': {'id': 209, 'name': 'Dr. Raphael E. Arku and Colleagues'}, 'isMobile': False, 'isMonitor': True, 'instruments': [{'id': 2, 'name': 'Government Monitor'}], 'sensors': [{'id': 6, 'name': 'pm10 µg/m³', 'parameter': {'id': 1, 'name': 'pm10', 'units': 'µg/m³', 'displayName': 'PM10'}}, {'id': 5, 'name': 'pm25 µg/m³', 'parameter': {'id': 2, 'name': 'pm25', 'units': 'µg/m³', 'displayName': 'PM2.5'}}], 'coordinates': {'latitude': 5.58389, 'longitude': -0.19968}, 'licenses': None, 'bounds': [-0.19968, 5.58389, -0.19968, 5.58389], 'distance': None, 'datetimeFirst': None, 'datetimeLast': None}, {'id': 4, 'name': 'NMT - Nima', 'locality': None, 'timezone': 'Africa/Acc

In [11]:
def json_para_dataframe(dados_brutos):
    estacoes = dados_brutos["results"]
    
    registros = []
    for estacao in estacoes:
        registros.append({
            "id":        estacao["id"],
            "nome":      estacao.get("name", "Sem nome"),
            "cidade":    estacao.get("locality", "Desconhecida"),
            "pais":      estacao["country"]["code"],
            "latitude":  estacao["coordinates"]["latitude"],
            "longitude": estacao["coordinates"]["longitude"],
        })
    
    return pd.DataFrame(registros)

df_estacoes = json_para_dataframe(dados_brutos)
df_estacoes.head()

,id,nome,cidade,pais,latitude,longitude
0,3,NMA - Nima,None,GH,5.583890,-0.199680
1,4,NMT - Nima,None,GH,5.581650,-0.198980
2,5,JTA - Jamestown,None,GH,5.540114,-0.210397
3,6,ADT - Asylum Down,None,GH,5.570722,-0.212056
4,7,ADEPA - Asylum Down,None,GH,5.567833,-0.204028


In [14]:
def buscar_estacoes_brasil(limite=10):
    url = f"{BASE_URL}/locations"
    
    parametros = {
        "countries_id": 31,  # ID do Brasil na OpenAQ
        "limit": limite
    }
    
    resposta = requests.get(url, headers=HEADERS, params=parametros)
    print(f"Status: {resposta.status_code}")
    return resposta.json()

dados_brasil = buscar_estacoes_brasil()
df_estacoes = json_para_dataframe(dados_brasil)
df_estacoes.head()

Status: 200


""


In [18]:
print(dados_brasil)

{'meta': {'name': 'openaq-api', 'website': '/', 'page': 1, 'limit': 10, 'found': 0}, 'results': []}


In [19]:
def buscar_id_brasil():
    url = f"{BASE_URL}/countries"
    
    parametros = {"limit": 100}
    
    resposta = requests.get(url, headers=HEADERS, params=parametros)
    dados = resposta.json()
    
    for pais in dados["results"]:
        if pais["code"] == "BR":
            print(f"ID do Brasil: {pais['id']}")
            print(f"Nome: {pais['name']}")
            return pais["id"]

id_brasil = buscar_id_brasil()

ID do Brasil: 45
Nome: Brazil


In [20]:
def buscar_estacoes_brasil(limite=10):
    url = f"{BASE_URL}/locations"
    
    parametros = {
        "countries_id": 45,  # ID correto do Brasil
        "limit": limite
    }
    
    resposta = requests.get(url, headers=HEADERS, params=parametros)
    print(f"Status: {resposta.status_code}")
    return resposta.json()

dados_brasil = buscar_estacoes_brasil()
df_estacoes = json_para_dataframe(dados_brasil)
df_estacoes.head()


Status: 200


,id,nome,cidade,pais,latitude,longitude
0,5012,Diadema,Diadema,BR,-23.685876,-46.611622
1,5220,Jacareí,Jacareí,BR,-23.294199,-45.968234
2,5222,Santa Gertrudes,Santa Gertrudes,BR,-22.459955,-47.536298
3,5231,Taubaté,Taubaté,BR,-23.032351,-45.575805
4,5233,Cid.Universitária-USP-Ipen,São Paulo,BR,-23.566342,-46.737414


In [21]:
def buscar_medicoes(location_id, limite=100):
    url = f"{BASE_URL}/locations/{location_id}/measurements"
    
    parametros = {"limit": limite}
    
    resposta = requests.get(url, headers=HEADERS, params=parametros)
    return resposta.json()

def processar_medicoes(dados_brutos):
    medicoes = dados_brutos.get("results", [])
    
    registros = []
    for m in medicoes:
        registros.append({
            "data":     m["period"]["datetimeTo"]["local"],
            "poluente": m["parameter"]["name"],
            "valor":    m["value"],
            "unidade":  m["parameter"]["units"]
        })
    
    df = pd.DataFrame(registros)
    df["data"] = pd.to_datetime(df["data"])
    df = df[df["valor"] >= 0]  # remove valores inválidos
    
    return df

# Pega a primeira estação brasileira da lista
primeiro_id = df_estacoes["id"].iloc[0]
nome_estacao = df_estacoes["nome"].iloc[0]

print(f"Buscando medições da estação: {nome_estacao}")

medicoes_brutas = buscar_medicoes(primeiro_id)
df_medicoes = processar_medicoes(medicoes_brutas)

df_medicoes.head(10)

Buscando medições da estação: Diadema


KeyError: 'data'

In [23]:
print(medicoes_brutas)

{'detail': 'Not Found'}


In [24]:
# Mostra todas as estações disponíveis
print(df_estacoes[["id", "nome", "cidade"]])

     id                        nome           cidade
0  5012                     Diadema          Diadema
1  5220                     Jacareí          Jacareí
2  5222             Santa Gertrudes  Santa Gertrudes
3  5231                     Taubaté          Taubaté
4  5233  Cid.Universitária-USP-Ipen        São Paulo
5  5239                      Osasco           Osasco
6  5243                   Pinheiros        São Paulo
7  5244                Paulínia-Sul              NaN
8  5252                   Congonhas        São Paulo
9  5254                       Tatuí            Tatuí


In [25]:
# Buscando medições de Osasco
medicoes_brutas = buscar_medicoes(5239)
print(medicoes_brutas)

{'detail': 'Not Found'}


In [26]:
def buscar_medicoes_v2(location_id, limite=100):
    url = f"{BASE_URL}/sensors"
    
    parametros = {
        "locations_id": location_id,
        "limit": limite
    }
    
    resposta = requests.get(url, headers=HEADERS, params=parametros)
    print(f"Status: {resposta.status_code}")
    dados = resposta.json()
    print(dados)
    return dados

medicoes_brutas = buscar_medicoes_v2(5239)

Status: 404
{'detail': 'Not Found'}


In [27]:
def buscar_ultimo_dado(location_id):
    url = f"{BASE_URL}/locations/{location_id}/latest"
    
    resposta = requests.get(url, headers=HEADERS)
    print(f"Status: {resposta.status_code}")
    return resposta.json()

# Testa com Osasco
dados_osasco = buscar_ultimo_dado(5239)
print(dados_osasco)

Status: 200
{'meta': {'name': 'openaq-api', 'website': '/', 'page': 1, 'limit': 100, 'found': 5}, 'results': [{'datetime': {'utc': '2023-04-05T19:00:00Z', 'local': '2023-04-05T16:00:00-03:00'}, 'value': 31.0, 'coordinates': {'latitude': -23.52672142, 'longitude': -46.79207766}, 'sensorsId': 13727, 'locationsId': 5239}, {'datetime': {'utc': '2023-04-05T20:00:00Z', 'local': '2023-04-05T17:00:00-03:00'}, 'value': 0.6, 'coordinates': {'latitude': -23.52672142, 'longitude': -46.79207766}, 'sensorsId': 13725, 'locationsId': 5239}, {'datetime': {'utc': '2023-04-05T20:00:00Z', 'local': '2023-04-05T17:00:00-03:00'}, 'value': 12.0, 'coordinates': {'latitude': -23.52672142, 'longitude': -46.79207766}, 'sensorsId': 13560, 'locationsId': 5239}, {'datetime': {'utc': '2023-04-05T19:00:00Z', 'local': '2023-04-05T16:00:00-03:00'}, 'value': 1.0, 'coordinates': {'latitude': -23.52672142, 'longitude': -46.79207766}, 'sensorsId': 13623, 'locationsId': 5239}, {'datetime': {'utc': '2023-04-05T20:00:00Z', 'lo

In [28]:
def buscar_sensores(location_id):
    """Busca todos os sensores de uma estação"""
    url = f"{BASE_URL}/locations/{location_id}/sensors"
    
    resposta = requests.get(url, headers=HEADERS)
    dados = resposta.json()
    
    sensores = []
    for s in dados["results"]:
        sensores.append({
            "sensor_id": s["id"],
            "poluente":  s["parameter"]["name"],
            "unidade":   s["parameter"]["units"]
        })
    
    return pd.DataFrame(sensores)

df_sensores = buscar_sensores(5239)
print(df_sensores)

   sensor_id poluente unidade
0      13727     pm10   µg/m³
1      13725       co     ppm
2      13560     pm25   µg/m³
3      13623      so2   µg/m³
4      13726      no2   µg/m³
